# 03 — Explore Simulated Sessions

Inspect the generated multi-speaker sessions.
- Load WAV + RTTM pairs
- Visualize speaker activity
- Verify no overlap, check turn gaps
- Listen to sample sessions

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import IPython.display as ipd

SPEAKER_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']

def parse_rttm(rttm_path):
    segments = []
    with open(rttm_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 9 and parts[0] == 'SPEAKER':
                segments.append({'speaker': parts[7], 'start': float(parts[3]), 'duration': float(parts[4])})
    return segments

print('Ready to explore simulated sessions.')

In [ ]:
# Visualize a session
sim_dir = '/data/simulated/librispeech/train_2spk'  # Change as needed
rttm_files = sorted(glob.glob(os.path.join(sim_dir, '*.rttm')))

if rttm_files:
    rttm_path = rttm_files[0]
    wav_path = rttm_path.replace('.rttm', '.wav')
    segments = parse_rttm(rttm_path)
    speakers = sorted(set(s['speaker'] for s in segments))
    
    audio, sr = sf.read(wav_path)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 5))
    
    time = np.arange(len(audio)) / sr
    ax1.plot(time, audio, linewidth=0.3, color='gray')
    ax1.set_ylabel('Amplitude')
    
    for seg in segments:
        idx = speakers.index(seg['speaker'])
        ax2.barh(idx, seg['duration'], left=seg['start'], height=0.6, 
                 color=SPEAKER_COLORS[idx % len(SPEAKER_COLORS)], alpha=0.8)
    ax2.set_yticks(range(len(speakers)))
    ax2.set_yticklabels(speakers)
    ax2.set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()
    
    # Listen
    ipd.display(ipd.Audio(wav_path))
else:
    print(f'No sessions found in {sim_dir}. Run script 07 first.')